In [1]:
# 安装依赖环境。 
# pymilvus用于操作milvus向量数据库；openai python库用于访问openAI API；requests：HTTP请求库；
# tqdm：进度条可视化库；torch：pytorch深度学习框架，常用于本地运行Embedding模型
!pip install "pymilvus[model]==2.5.10" openai==1.82.0 requests==2.32.3 tqdm==4.67.1 torch==2.7.0

In [17]:
import os
api_key = os.getenv('DEEPSEEK_API_KEY')

In [5]:
# 准备原始的文本数据

from glob import glob
text_lines = []

for file_path in glob("../mfd.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()
    text_lines += file_text.split("# ")

len(text_lines)

30

In [6]:
# 准备LLM
from openai import OpenAI
deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
)

In [7]:
# 准备Embedding模型
from pymilvus import model as milvus_model

embedding_model = milvus_model.DefaultEmbeddingFunction()

# 使用刚创建的embedding模型，测试一个文本，并打印其纬度和前几个元素
test_embedding = embedding_model.encode_queries(["I am making my second rag by using milvus and deepseek"])[0]
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

/opt/anaconda3/envs/deepseek-p/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


768
[-0.03339131 -0.00667745  0.01231828  0.00443335  0.00936556 -0.01949256
 -0.07040836  0.01361133  0.02410534 -0.01485614]


In [8]:
# 创建collection
from pymilvus import MilvusClient
milvus_client = MilvusClient(uri="./milvus_law_demo.db")
collection_name = "second_rag_collection"

if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim, # 维度数
    metric_type="IP", # 内积距离，IP表示值越大越相似
    consistency_level="Strong", #一致性级别：Strong总是读到最新数据，可能稍慢。
)

# 插入数据
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating law embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Creating law embeddings: 100%|██████████| 30/30 [00:00<00:00, 343795.41it/s]


{'insert_count': 30, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29], 'cost': 0}

In [16]:
# 检索查询数据
question1 = "简要描述下，民法典是一部关于什么的法律法规?"
question2 = "关于不动产变更，有什么规定？"

search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries([question1, question2]), # 将问题question转为查询向量
    limit=3, # 返回最相似的top3结果
    search_params={"metric_type": "IP", "params":{}}, # 内积距离
    output_fields=["text"], # 返回text字段
)

# 查询的搜索结果
import json
retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, ensure_ascii=False, indent=4))


# 使用LLM获取RAG响应

context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)


SYSTEM_PROMPT = """
Human: 你是一个AI助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question>标签括起来的问题。
<context>
{context}
<context>
<question>
{question1}
</question>
<question>
{question2}
</question>
"""

response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ]
)

print(response.choices[0].message.content)


[
    [
        "二、权利质权\n\n**第四百四十九条** 可以出质的权利包括：\n（一）汇票、本票、支票；\n（二）债券、存款单；\n（三）仓单、提单；\n（四）可以转让的基金份额、股权；\n（五）可以转让的注册商标专用权、专利权、著作权等知识产权中的财产权；\n（六）应收账款；\n（七）法律、行政法规规定可以出质的其他财产权利。\n\n**第四百五十条** 以汇票、本票、支票、债券、存款单、仓单、提单出质的，当事人应当订立书面合同。质权自权利凭证交付之日起设立。\n\n**第四百五十一条** 以记名股票出质的，当事人应当订立书面合同。质权自股票交付之日起设立。\n以未上市公司股权出质的，适用公司法有关股权转让的规定。\n\n**第四百五十二条** 以可以转让的基金份额、股权出质的，当事人应当订立书面合同。质权自基金份额、股权登记于证券登记结算机构或者公司章程载明的股权登记簿时设立。\n以未上市公司股权出质的，适用公司法有关股权转让的规定。\n\n**第四百五十三条** 以可以转让的注册商标专用权、专利权、著作权等知识产权中的财产权出质的，当事人应当订立书面合同。质权自权利质押登记于相关部门时设立。\n\n**第四百五十四条** 以应收账款出质的，当事人应当订立书面合同。质权自应收账款质押登记于中国人民银行征信中心时设立。\n\n**第四百五十五条** 以法律、行政法规规定可以出质的其他财产权利出质的，依照法律、行政法规的规定。\n\n**第四百五十六条** 权利质权除适用本节规定外，参照适用本章动产质权的有关规定。\n\n####",
        0.5430307984352112
    ],
    [
        "第三章 合同的变更和转让\n\n**第五百四十八条** 当事人协商一致，可以变更合同。\n\n**第五百四十九条** 当事人对合同变更的内容约定不明确的，推定为未变更。\n\n**第五百五十条** 债权人可以将合同的权利全部或者部分转让给第三人，但是有下列情形之一的除外：\n（一）根据合同性质不得转让；\n（二）按照当事人约定不得转让；\n（三）依照法律规定不得转让。\n债权人转让权利的，应当通知债务人。未经通知，该转让对债务人不发生效力。\n\n**第五百五十一条** 债权人转让权利的，受让人取得与债权有关的从权利